### Phase 2 prediction

- Dataset structure:

    ```json
    {
        "system_params": {
            "temperature": float,
            "top_k": int,
            "repetition_penalty": float,
            "max_seq_len": int
        },
        "model_arch": {
            "num_layers": int,
            "num_heads": int
        },
        "samples": [
            {
                "layer": int,
                "head": int,
                "attention_matrix": np.array(seq_len, seq_len),
                "seq_pos": int,
            },
        ],
        "label": {
            "remaining_tokens": int,
            "over_max_seq_len": bool
        }
    }
    ```
- Task: predict the remaining tokens (and over_max_seq_len) for each sample in the dataset.
  - Regression task
  - Models can be used: MLP, Transformer, RNN, LSTM, GRU, etc.

Here we use **MLP** and train it with the dataset.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ======================
# Dataset Preparation
# ======================
class OutputLengthDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features  # Preprocessed features (e.g., flattened attention matrices)
        self.labels = labels      # remaining_tokens

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input": torch.tensor(self.features[idx], dtype=torch.float32),
            "label": torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# Load and preprocess data (example)
# Replace with actual data loading
features = [...]  # Preprocessed features (e.g., flattened attention matrices)
labels = [...]    # remaining_tokens

# Split data
X_train, X_val, y_train, y_val = train_test_split(features, labels, test_size=0.2)

# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# Create datasets and dataloaders
train_dataset = OutputLengthDataset(X_train, y_train)
val_dataset = OutputLengthDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

# ======================
# Model Definition
# ======================
class LightweightMLP(nn.Module):
    def __init__(self, input_dim):
        super(LightweightMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)  # Output: remaining_tokens

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# Initialize model, loss, and optimizer
input_dim = X_train.shape[1]
model = LightweightMLP(input_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ======================
# Training Loop
# ======================
def train_model(model, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            inputs = batch["input"]
            labels = batch["label"].unsqueeze(-1)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch["input"]
                labels = batch["label"].unsqueeze(-1)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss/len(train_loader):.4f} - Val Loss: {val_loss/len(val_loader):.4f}")

# Train the model
train_model(model, train_loader, val_loader, epochs=20)